# Stanford Battery Cycle Life Dataset — Ingestion Notebook

**Dataset:** Severson et al. 2019 — *Data-driven prediction of battery cycle life before capacity degradation*  
**Cells:** 140 A123 LFP/graphite 18650 cells, 3 batches, various fast-charge protocols  
**Format:** Per-cycle summary — one row per cycle per cell

### What this notebook does
1. Loads and inspects all three CSV files
2. Explores the dataset — capacity fade, IR growth, protocol spread
3. Prepares per-cell files and config files
4. Loads all 140 cells directly into battdb via psycopg2
5. Verifies the data landed correctly

### Prerequisites
- battdb running on `localhost:5454` (Docker)
- `.env` file with DB credentials
- `pip install psycopg2-binary pandas matplotlib tqdm python-dotenv`

## 1. Imports & config

In [ ]:
import os
import json
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import psycopg2
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.3})

# ── paths ──────────────────────────────────────────────────────────────
DATA_DIR  = r"C:\Users\Vcpat\Downloads\data"
CELLS_DIR = r"C:\Users\Vcpat\Downloads\data\cells"
ENV_PATH  = r"C:\Users\Vcpat\Downloads\battetl-main\battetl-main\.env"

CSV_FILES = {
    "full":  os.path.join(DATA_DIR, "Lithium-Ion Battery Cycle Life.csv"),
    "100cy": os.path.join(DATA_DIR, "100_Cycle_Lithium-Ion Battery Cycle Life.csv"),
    "50cy":  os.path.join(DATA_DIR, "50_Cycle_Lithium-Ion Battery Cycle Life.csv"),
}

load_dotenv(ENV_PATH, override=True)

print("Paths configured.")
print("Files found:", [os.path.basename(v) for v in CSV_FILES.values() if os.path.exists(v)])

## 2. Load & inspect the data

In [ ]:
df = pd.read_csv(CSV_FILES["full"])

batch_pattern = r'(b\d+)'
print(f"Shape: {df.shape}")
print(f"Unique batteries: {df['battery_id'].nunique()}")
print(f"Batches: {sorted(df['battery_id'].str.extract(batch_pattern)[0].unique())}")
print(f"Total cycles across all cells: {len(df):,}")
print()
df.head()

In [ ]:
df.describe().round(4)

In [ ]:
cell_summary = df.groupby('battery_id').agg(
    cycle_life   = ('cycle_life', 'first'),
    total_cycles = ('cycle', 'count'),
    C1           = ('C1', 'first'),
    Q1           = ('Q1', 'first'),
    C2           = ('C2', 'first'),
    batch        = ('battery_id', lambda x: x.iloc[0][:2])
).reset_index()

print(f"Cycle life range: {cell_summary['cycle_life'].min():.0f} - {cell_summary['cycle_life'].max():.0f} cycles")
print(f"Mean cycle life:  {cell_summary['cycle_life'].mean():.0f} cycles")
cell_summary.head(10)

## 3. Explore the dataset
### 3a. Cycle life distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cell_summary['cycle_life'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Cycle life')
axes[0].set_ylabel('Number of cells')
axes[0].set_title('Cycle life distribution (140 cells)')

for batch, grp in cell_summary.groupby('batch'):
    axes[1].scatter(grp.index, grp['cycle_life'], label=f'Batch {batch}', alpha=0.7, s=40)
axes[1].set_xlabel('Cell index')
axes[1].set_ylabel('Cycle life')
axes[1].set_title('Cycle life by batch')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3b. Capacity fade — all cells

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

vmin = cell_summary['cycle_life'].min()
vmax = cell_summary['cycle_life'].max()
cmap = cm.get_cmap('RdYlGn')

for _, row in cell_summary.iterrows():
    cell_data = df[df['battery_id'] == row['battery_id']]
    colour = cmap((row['cycle_life'] - vmin) / (vmax - vmin))
    axes[0].plot(cell_data['cycle'], cell_data['QD'], color=colour, alpha=0.4, linewidth=0.6)
    axes[1].plot(cell_data['cycle'], cell_data['IR'], color=colour, alpha=0.4, linewidth=0.6)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])

axes[0].set_xlabel('Cycle number')
axes[0].set_ylabel('Discharge capacity (Ah)')
axes[0].set_title('Capacity fade — all 140 cells')
plt.colorbar(sm, ax=axes[0], label='Cycle life')

axes[1].set_xlabel('Cycle number')
axes[1].set_ylabel('Internal resistance (Ohm)')
axes[1].set_title('IR growth — all 140 cells')
plt.colorbar(sm, ax=axes[1], label='Cycle life')

plt.tight_layout()
plt.show()
print("Green = long life, Red = short life")

### 3c. Effect of charge protocol on cycle life

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(cell_summary['C1'], cell_summary['cycle_life'], alpha=0.6, s=40, color='steelblue')
axes[0].set_xlabel('C1 rate (C)')
axes[0].set_ylabel('Cycle life')
axes[0].set_title('C1 rate vs cycle life')

axes[1].scatter(cell_summary['Q1'], cell_summary['cycle_life'], alpha=0.6, s=40, color='coral')
axes[1].set_xlabel('Q1 - SOC% at end of step 1')
axes[1].set_title('Q1 vs cycle life')

axes[2].scatter(cell_summary['C2'], cell_summary['cycle_life'], alpha=0.6, s=40, color='seagreen')
axes[2].set_xlabel('C2 rate (C)')
axes[2].set_title('C2 rate vs cycle life')

plt.tight_layout()
plt.show()

### 3d. Single cell deep-dive

In [ ]:
CELL_ID = 'b1c0'  # change to inspect any cell

cell_data = df[df['battery_id'] == CELL_ID].copy()
meta = cell_summary[cell_summary['battery_id'] == CELL_ID].iloc[0]

print(f"Cell: {CELL_ID}")
print(f"Cycle life: {int(meta['cycle_life'])} cycles")
print(f"Protocol:   C1={meta['C1']}C for {meta['Q1']}% SOC, then C2={meta['C2']}C")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'{CELL_ID} — cycle life {int(meta["cycle_life"])} cycles', fontsize=13)

axes[0,0].plot(cell_data['cycle'], cell_data['QD'], color='steelblue')
axes[0,0].set_ylabel('Discharge capacity (Ah)')
axes[0,0].set_title('Capacity fade')

axes[0,1].plot(cell_data['cycle'], cell_data['IR'], color='coral')
axes[0,1].set_ylabel('Internal resistance (Ohm)')
axes[0,1].set_title('IR growth')

axes[1,0].plot(cell_data['cycle'], cell_data['Tavg'], color='seagreen', label='Avg')
axes[1,0].fill_between(cell_data['cycle'], cell_data['Tmin'], cell_data['Tmax'], alpha=0.2, color='seagreen')
axes[1,0].set_xlabel('Cycle')
axes[1,0].set_ylabel('Temperature (C)')
axes[1,0].set_title('Temperature (min/avg/max)')
axes[1,0].legend()

axes[1,1].plot(cell_data['cycle'], cell_data['chargetime'], color='purple')
axes[1,1].set_xlabel('Cycle')
axes[1,1].set_ylabel('Charge time (min)')
axes[1,1].set_title('Charge time')

for ax in axes.flat:
    if ax.get_xlabel() == '':
        ax.set_xlabel('Cycle')

plt.tight_layout()
plt.show()

## 4. Prepare per-cell files

In [ ]:
os.makedirs(CELLS_DIR, exist_ok=True)

for battery_id, group in df.groupby('battery_id'):
    cell_dir = os.path.join(CELLS_DIR, battery_id)
    os.makedirs(cell_dir, exist_ok=True)
    cell_df = group.drop(columns=['battery_id', 'cycle_life', 'C1', 'Q1', 'C2'])
    cell_df.to_csv(os.path.join(cell_dir, f"{battery_id}_cycles.csv"), index=False)

print(f"Created {df['battery_id'].nunique()} cell directories under {CELLS_DIR}")
print("Example files:", os.listdir(os.path.join(CELLS_DIR, 'b1c0')))

## 5. Connect to battdb

Make sure Docker is running and `docker compose up -d` has been run.

In [ ]:
conn = psycopg2.connect(
    host='localhost',
    port=5454,
    dbname='battdb',
    user='postgres',
    password='password'
)
print('Connected to battdb')

## 6. Direct insertion function

battetl's extractor only handles Arbin/Maccor native formats, so we bypass it and insert directly using psycopg2.

Column mapping:
- `QC` (Ah) x1000 → `reported_charge_capacity_mah`
- `QD` (Ah) x1000 → `reported_discharge_capacity_mah`
- `chargetime` (min) x60 → `reported_charge_time_s`
- `IR`, `Tavg`, `Tmin`, `Tmax` → `other_details` (jsonb)

In [ ]:
def insert_cell_direct(conn, battery_id, cell_df, cell_summary):
    cur = conn.cursor()
    row = cell_summary[cell_summary['battery_id'] == battery_id].iloc[0]
    batch = battery_id.split('c')[0]
    c1 = row['C1'] if pd.notna(row['C1']) else 0
    q1 = row['Q1'] if pd.notna(row['Q1']) else 0
    c2 = row['C2'] if pd.notna(row['C2']) else 0

    # 1. Upsert cells_meta
    cur.execute("""
        INSERT INTO cells_meta (manufacturer, manufacturer_pn, form_factor, capacity_mah, chemistry)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (manufacturer_pn) DO NOTHING
        RETURNING cell_type_id
    """, ('A123 Systems', 'APR18650M1A', '18650', 1100, 'LFP/graphite'))
    result = cur.fetchone()
    if not result:
        cur.execute("SELECT cell_type_id FROM cells_meta WHERE manufacturer_pn = 'APR18650M1A'")
        result = cur.fetchone()
    cell_type_id = result[0]

    # 2. Upsert cell
    cur.execute("""
        INSERT INTO cells (manufacturer_sn, cell_type_id, batch_number)
        VALUES (%s, %s, %s)
        ON CONFLICT (manufacturer_sn) DO UPDATE SET batch_number = EXCLUDED.batch_number
        RETURNING cell_id
    """, (battery_id, cell_type_id, batch))
    cell_id = cur.fetchone()[0]

    # 3. Upsert test_meta
    cur.execute("""
        INSERT INTO test_meta (test_name, cell_id, channel, comments)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (test_name) DO UPDATE SET cell_id = EXCLUDED.cell_id
        RETURNING test_id
    """, (
        f"Stanford_CycleLife_{battery_id}",
        cell_id,
        1,
        f"Severson et al. 2019. Protocol: {c1}C to {q1}% SOC, then {c2}C."
    ))
    test_id = cur.fetchone()[0]

    # 4. Insert cycle_stats rows
    rows = [
        (
            test_id,
            int(r['cycle']),
            round(r['QC'] * 1000, 4),
            round(r['QD'] * 1000, 4),
            round(r['chargetime'] * 60, 1),
            json.dumps({
                'IR_ohm': round(float(r['IR']), 6),
                'Tavg_c': round(float(r['Tavg']), 4),
                'Tmin_c': round(float(r['Tmin']), 4),
                'Tmax_c': round(float(r['Tmax']), 4),
            })
        )
        for _, r in cell_df.iterrows()
    ]

    cur.executemany("""
        INSERT INTO test_data_cycle_stats (
            test_id, cycle,
            reported_charge_capacity_mah,
            reported_discharge_capacity_mah,
            reported_charge_time_s,
            other_details
        ) VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT DO NOTHING
    """, rows)

    conn.commit()
    cur.close()
    return test_id, len(rows)


# Test on b1c0 first
test_cell_id = 'b1c0'
test_cell_df = pd.read_csv(os.path.join(CELLS_DIR, test_cell_id, f"{test_cell_id}_cycles.csv"))
test_id, n = insert_cell_direct(conn, test_cell_id, test_cell_df, cell_summary)
print(f"b1c0: test_id={test_id}, {n} cycles inserted")

## 7. Load all 140 cells

In [ ]:
from tqdm.notebook import tqdm

results = {'success': [], 'failed': []}

for battery_id, group in tqdm(df.groupby('battery_id'), desc='Loading cells'):
    csv_path = os.path.join(CELLS_DIR, battery_id, f"{battery_id}_cycles.csv")
    cell_csv = pd.read_csv(csv_path)
    try:
        test_id, n = insert_cell_direct(conn, battery_id, cell_csv, cell_summary)
        results['success'].append(battery_id)
    except Exception as e:
        conn.rollback()
        results['failed'].append((battery_id, str(e)))
        print(f"  {battery_id}: {e}")

print(f"\nSuccess: {len(results['success'])} cells")
print(f"Failed:  {len(results['failed'])} cells")
if results['failed']:
    for cid, err in results['failed']:
        print(f"  {cid}: {err}")

## 8. Verify — query battdb

In [ ]:
cur = conn.cursor()

# How many tests landed?
cur.execute("SELECT COUNT(*) FROM test_meta WHERE test_name LIKE 'Stanford_CycleLife_%'")
print(f"Tests in battdb: {cur.fetchone()[0]}")

# How many cycle stats rows?
cur.execute("""
    SELECT COUNT(*) FROM test_data_cycle_stats cs
    JOIN test_meta tm ON cs.test_id = tm.test_id
    WHERE tm.test_name LIKE 'Stanford_CycleLife_%'
""")
print(f"Total cycle rows: {cur.fetchone()[0]:,}")

cur.close()

In [ ]:
# Read b1c0 back and plot
query = """
    SELECT cs.cycle,
           cs.reported_discharge_capacity_mah,
           cs.reported_charge_capacity_mah,
           cs.reported_charge_time_s,
           cs.other_details
    FROM test_data_cycle_stats cs
    JOIN test_meta tm ON cs.test_id = tm.test_id
    WHERE tm.test_name = 'Stanford_CycleLife_b1c0'
    ORDER BY cs.cycle
"""
db_df = pd.read_sql(query, conn)
db_df['IR_ohm']  = db_df['other_details'].apply(lambda x: x['IR_ohm'] if isinstance(x, dict) else json.loads(x)['IR_ohm'])
db_df['Tavg_c']  = db_df['other_details'].apply(lambda x: x['Tavg_c'] if isinstance(x, dict) else json.loads(x)['Tavg_c'])

print(f"Rows returned: {len(db_df)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('b1c0 — read back from battdb', fontsize=13)

axes[0].plot(db_df['cycle'], db_df['reported_discharge_capacity_mah'], color='steelblue')
axes[0].set_xlabel('Cycle')
axes[0].set_ylabel('Discharge capacity (mAh)')
axes[0].set_title('Capacity fade')

axes[1].plot(db_df['cycle'], db_df['IR_ohm'], color='coral')
axes[1].set_xlabel('Cycle')
axes[1].set_ylabel('Internal resistance (Ohm)')
axes[1].set_title('IR growth')

plt.tight_layout()
plt.show()

In [ ]:
conn.close()
print('Connection closed. All done.')